# Phase 2 — Fine-tune **Gemma 4-E4B** on Arabic WSD (Dataset A / El-Razzaz)

Same pipeline as Phase 1, **changing only the base model**, so the Gemma 2-2B vs Gemma 4-E4B comparison is fair.
Gemma 4 was released 2026-04-02; we use the **E4B** (4B-class, edge/laptop) variant, **text only**.

**Before you run:**
1. `Runtime → GPU (T4)` and set `REPO_URL` to your fork (same as Phase 1).
2. Confirm the exact Unsloth Gemma-4 4-bit id with the discovery cell below.
3. **T4 + 4B can OOM** — if so, set `WSD_MAX_SEQ_LEN=768` in the Config cell (or use Colab Pro/L4).

In [ ]:
# 0. Confirm we have a GPU
!nvidia-smi

In [ ]:
# 1. Install dependencies (use the LATEST unsloth/transformers for Gemma 4 support)
!pip install -q -U unsloth trl peft transformers datasets bitsandbytes accelerate scikit-learn python-dotenv

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. Clone your fork (with the fixed scripts)
REPO_URL = 'https://github.com/RokayaAlHarakeh/Arabic-WSD-LLM.git'  # your fork
REPO_DIR = '/content/Arabic-WSD-LLM'
BRANCH   = 'phase1-gemma2-2b-datasetA'  # branch with the fixed scripts (use 'main' if you merged)

import os, shutil
assert '<your-username>' not in REPO_URL, 'Set REPO_URL to your pushed fork first.'
if os.path.isdir(REPO_DIR):
    shutil.rmtree(REPO_DIR)
!git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO_DIR}
SCRIPT_DIR = os.path.join(REPO_DIR, 'Gemma', 'Fine-tuning', 'Dataset-A')
print('Scripts:', os.listdir(SCRIPT_DIR))

In [ ]:
# 4. Stage Dataset A into Drive at PROJECT_DIR/data
PROJECT_DIR = '/content/drive/MyDrive/WSD_Project'
DATA_DIR = os.path.join(PROJECT_DIR, 'data')
os.makedirs(DATA_DIR, exist_ok=True)

src_ds = os.path.join(REPO_DIR, 'Datasets', 'Dataset-A')
for f in ['test_set.json','test_truth.json','test_dictionary.json',
          'train80_set.json','train80_truth.json','train80_dictionary.json']:
    shutil.copy(os.path.join(src_ds, f), os.path.join(DATA_DIR, f))
shutil.copy(os.path.join(SCRIPT_DIR, 'fine_tuning_dataset_elrazzaz.jsonl'),
            os.path.join(DATA_DIR, 'fine_tuning_dataset_elrazzaz.jsonl'))
print('Staged:', os.listdir(DATA_DIR))

In [ ]:
# 4b. DISCOVERY: list Unsloth's 4-bit Gemma-4 checkpoints, then pick the E4B id below.
from huggingface_hub import list_models
for m in list_models(author='unsloth', search='gemma-4', limit=50):
    print(m.id)

## Config — the only cell that differs from Phase 1

In [ ]:
os.environ['WSD_PROJECT_DIR'] = PROJECT_DIR
# Use the exact E4B id from the discovery cell (e.g. 'unsloth/gemma-4-e4b-it' or a -bnb-4bit variant):
os.environ['WSD_BASE_MODEL']  = 'unsloth/gemma-4-e4b-it'   # <-- confirm against discovery output
os.environ['WSD_MODEL_TAG']   = 'gemma4_e4b'
os.environ['WSD_MAX_SEQ_LEN'] = '1024'                     # lower to '768' if the T4 OOMs
# from google.colab import userdata; os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print('Config set for', os.environ['WSD_MODEL_TAG'], '->', os.environ['WSD_BASE_MODEL'])

In [ ]:
# 5. Fine-tune (LoRA, 4-bit)
!python {SCRIPT_DIR}/finetuning.py

In [ ]:
# 6. Inference over the test set
!python {SCRIPT_DIR}/infer_model.py

In [ ]:
# 7. Evaluate
!python {SCRIPT_DIR}/eval.py

In [ ]:
# 8. Sanity check: report + a few raw predictions
import json
tag = os.environ['WSD_MODEL_TAG']
out = os.path.join(PROJECT_DIR, 'outputs', tag)
print('REPORT:', json.dumps(json.load(open(os.path.join(out, f'report_{tag}.json'))), indent=2))
print('\n--- first 1500 chars of debug log ---')
print(open(os.path.join(out, f'debug_{tag}.txt'), encoding='utf-8').read()[:1500])